# West-of-N

```{note}
$\text{Best-of-N} + \text{Worst-of-N} = \text{West-of-N}$
```

## Self-Training for Preference Modeling

Assume access to some initial preference dataset $\mathcal{D}_{L} = \{(x,y_{+},y_{-}): y_{+}\succ y_{-}\}$, we use this data to train a base preference model parameterized by $\theta$: let $P_{\theta}(y_{+}\succ y_{-}|x)$
model the probability of response $y_{+}$ being preferred over
$y_{-}$ for a query $x$.

A simple strategy to generate synthetic preference data for
unlabeled query $x$ is to sample two responses $y_1$, $y_2$ from the
generation policy $\pi(x)$, and to pseudo-label the preference pair based on $P_{\theta}(y_1\succ y_2|x)$:

$$
f_{\pm}(x) = 
\begin{cases}
(y_1,y_2)\quad\text{if}\quad P_{\theta}(y_1\succ y_2|x)>0.5\\
(y_2,y_1)\quad\text{else}
\end{cases}
$$

This approach can be used to generate a pseudo-preference
dataset $\mathcal{D}_{L'}$ , and a self-trained student reward model, parameterized
by $\theta '$, can be optimized on $\mathcal{D}_{L}\cup\mathcal{D}_{L'}$.

## West-of-N Self-Training

We propose to maximize the probability
of correctly labeling a pair of on-policy responses to a given
query, according to the base preference model:

$$
\underset{(y_{+},y_{-})\sim\pi(x)}{\max}P_{\theta}(y_+\succ y_-|x)
$$

In practice, this objective can be approximated by sampling
a pool of $N$ candidate outputs from the policy and identify
the best- and worst-scored ones.

````{prf:theorem}
Let $P^{\ast}(y_{+}\succ y_{-}|x)$ denote
the ground-truth preference function to be approximated. Assume $|P_{\theta}(y_{+}\succ y_{-}|x) - P^{\ast}(y_{+}\succ y_{-}|x)| <\epsilon$ for all $(x,y_{+},y_{-})$. For any $x$, the West-of-N preference pair $f_{\pm}(x)=(y_{+},y_{-})$ is correctly labeled
with probability $P^{\ast}(y_{+}\succ y_{-}|x)\ge 1-2\epsilon$.
````

## Pseudo-Preference Filtering

To further improve the
quality of generated preference pairs, these can be filtered
based on the confidence of their preference label and
their coverage of the relevant response distribution.

We
measure model confidence in labeling a preference through
the prediction $P_{\theta}(y_+\succ y_-|x)$, and only retain West-of-N
pairs above a certain quantile. Similarly, we also apply a
likelihood threshold of both positive and negative responses $\pi(y_{+}|x)$ and $\pi(y_{-}|x)$, to ensure the responses being compared
remain in-distribution. We determine final threshold
values through validation performance.